# Image Captioning Using ViT and LSTM

### I. Project Configuration

In [1]:
import os
import re
import csv
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
# from transformers import ViTModel, ViTConfig, ViTFeatureExtractor
from transformers import ViTModel, AutoImageProcessor
from collections import Counter, defaultdict
from PIL import Image
import numpy as np
import random
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from typing import List, Tuple
import math
import copy
import pdb
import pickle

c:\Users\PC\anaconda3\envs\main-gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EMBED_DIM = 256
HIDDEN_DIM = 512
# LEARNING_RATE = 0.0005
LEARNING_RATE_ENCODER = 1e-5
LEARNING_RATE_DECODER = 5e-4
WEIGHT_DECAY_ENCODER = 1e-4
WEIGHT_DECAY_DECODER = 1e-4
BATCH_SIZE = 64
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3
MIN_WORD_FREQ = 5
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
NUM_WORKERS = 0

DATA_FOLDER_PATH = "../data"
IMAGE_FOLDER_PATH = f"{DATA_FOLDER_PATH}/Images"
CAPTION_FILE_PATH = f"{DATA_FOLDER_PATH}/captions.txt"

ENCODER = "facebook/vit-mae-base"

BEST_CHECKPOINT_PATH = "../models/vit_lstm_best-v4.pth"
FINAL_MODEL_PATH = "../models/vit_lstm_final-v4.pth"
VOCAB_PATH = "../models/vocab-v4.pkl"

RESUME = True

Using device: cuda


In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
class Vocabulary:
    def __init__(self, freq_threshold = 5):
        self.freq_threshold = freq_threshold
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {v: k for k, v in self.itos.items()}
        self.index = 4

    def __len__(self):
        return len(self.itos)
    
    def tokenizer(self, text):
        text = text.lower()
        tokens = re.findall(r"\w+", text)
        return tokens
    
    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        for sentence in sentence_list:
            tokens = self.tokenizer(sentence)
            frequencies.update(tokens)
        
        for word, freq in frequencies.items():
            if freq >= self.freq_threshold:
                self.stoi[word] = self.index
                self.itos[self.index] = word
                self.index += 1

    def numericalize(self, text):
        tokens = self.tokenizer(text)
        numericalized = []
        for token in tokens:
            if token in self.stoi:
                numericalized.append(self.stoi[token])
            else:
                numericalized.append(self.stoi["<UNK>"])

        numericalized = [self.stoi["<SOS>"]] + numericalized + [self.stoi["<EOS>"]]
        return numericalized

In [5]:
class FlickrDataset(Dataset):
    def __init__(self, imgid2captions, vocab, transform = None):
        self.imgid2captions = []
        self.transform = transform
        self.vocab = vocab

        for img_id, captions in imgid2captions.items():
            for caption in captions:
                self.imgid2captions.append((img_id, caption))

    def __len__(self):
        return len(self.imgid2captions)
    
    def __getitem__(self, index):
        img_id, caption = self.imgid2captions[index]
        img_path = os.path.join(IMAGE_FOLDER_PATH, img_id)
        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        # numericalized_caption = [self.vocab.stoi["<SOS>"]]
        numericalized_caption = self.vocab.numericalize(caption)
        # numericalized_caption.append(self.vocab.stoi["<EOS>"])

        return image, torch.tensor(numericalized_caption, dtype = torch.long)

In [6]:
def parse_tokens(csv_file):
    imgid2captions = {}
    with open(csv_file, "r", encoding = "utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) < 2:
                continue

            img_id, caption = row[0].strip(), row[1].strip()
            if not caption:
                continue

            if img_id not in imgid2captions:
                imgid2captions[img_id] = []

            imgid2captions[img_id].append(caption)

    return imgid2captions    

In [7]:
def collate_fn(batch):
    batch.sort(key = lambda x: len(x[1]), reverse = True)
    
    images = [item[0] for item in batch]
    captions = [item[1] for item in batch]
    lengths = [len(caption) for caption in captions]
    max_len = max(lengths)
    padded_captions = torch.zeros(len(captions), max_len, dtype = torch.long)

    for i, caption in enumerate(captions):
        end = lengths[i]
        padded_captions[i, :end] = caption[:end]
    images = torch.stack(images, dim = 0)

    return images, padded_captions, lengths

### II. Model Definition

In [8]:
class Encoder(nn.Module):
    def __init__(self, embed_dim, freeze = False):
        super().__init__()
        self.vit = ViTModel.from_pretrained(ENCODER)

        for i, block in enumerate(self.vit.encoder.layer):
            for param in block.parameters():
                param.requires_grad = (i >= 8) and not freeze


        self.linear = nn.Sequential(
            nn.Linear(self.vit.config.hidden_size, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim, embed_dim),
            nn.LayerNorm(embed_dim)
        )

    def forward(self, images):
        vit_outputs = self.vit(pixel_values = images)
        patch_embeddings = vit_outputs.last_hidden_state[:, 1:, :]
        features = self.linear(patch_embeddings)
        return features

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, encoder_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim
        self.head_dim = hidden_dim // num_heads

        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"

        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key = nn.Linear(encoder_dim, hidden_dim)
        self.value = nn.Linear(encoder_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, encoder_dim)

    def forward(self, hidden, encoder_outputs):
        B, N, _ = encoder_outputs.shape

        Q = self.query(hidden).view(B, self.num_heads, self.head_dim) 
        K = self.key(encoder_outputs).view(B, N, self.num_heads, self.head_dim).transpose(1, 2) 
        V = self.value(encoder_outputs).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q.unsqueeze(2), K.transpose(-2, -1)) / (self.head_dim ** 0.5)  
        attn = torch.softmax(scores, dim = -1) 

        context = torch.matmul(attn, V) 
        context = context.transpose(1, 2).contiguous().view(B, self.hidden_dim)

        return self.fc_out(context) 

class Decoder(nn.Module):
    def __init__(self, embed_dim, hidden_dim, vocab_size, encoder_dim = 256, num_layers = 2, dropout = 0.3, num_heads = 4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(embed_dim + encoder_dim, hidden_dim, num_layers, batch_first = True, dropout = dropout if num_layers > 1 else 0)
        self.attention = MultiHeadAttention(hidden_dim, encoder_dim, num_heads = num_heads)

        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, encoder_outputs, captions):
        captions_in = captions[:, :-1] 
        emb = self.dropout(self.embedding(captions_in)) 

        batch_size, seq_len, _ = emb.size()
        h, c = (torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size, device=emb.device),
                torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size, device=emb.device))

        outputs = []
        for t in range(seq_len):
            context = self.attention(h[-1], encoder_outputs)
            lstm_input = torch.cat((emb[:, t], context), dim = 1).unsqueeze(1) 
            out, (h, c) = self.lstm(lstm_input, (h, c))
            outputs.append(self.fc(out.squeeze(1)))

        outputs = torch.stack(outputs, dim = 1)
        return outputs

In [10]:
class Model(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images, captions):
        features = self.encoder(images)
        outputs = self.decoder(features, captions)
        return outputs

### III. Training Pipeline

In [11]:
def train_one_epoch(model, dataloader, criterion, optimizer, vocab_size, epoch):
    model.train()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc = f"Epoch {epoch+1}", unit = "batch")
    for images, captions, _lengths in progress_bar:
        images = images.to(DEVICE)
        captions = captions.to(DEVICE)
 
        optimizer.zero_grad()
        outputs = model(images, captions)

        outputs = outputs[:, :captions.size(1)-1, :].contiguous().view(-1, vocab_size)
        targets = captions[:, 1:].contiguous().view(-1)
 
        mask = targets != 0
        loss = criterion(outputs[mask], targets[mask])
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 5.0)
        optimizer.step()
 
        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    avg_loss = total_loss / len(dataloader)
    
    return avg_loss

In [12]:
def validate(model, dataloader, criterion, vocab_size):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, captions, _lengths in dataloader:
            images = images.to(DEVICE)
            captions = captions.to(DEVICE)

            outputs = model(images, captions)
            outputs = outputs[:, :captions.size(1)-1, :].contiguous().view(-1, vocab_size)

            targets = captions[:, 1:].contiguous().view(-1)
            mask = targets != 0
            loss = criterion(outputs[mask], targets[mask])

            loss = criterion(outputs, targets)
            total_loss += loss.item()
    avg_val_loss = total_loss / len(dataloader)
    return avg_val_loss

In [42]:
def precook(s, n=4, out=False):
    """
    Takes a string as input and returns an object that can be given to
    either cook_refs or cook_test. This is optional: cook_refs and cook_test
    can take string arguments as well.
    :param s: string : sentence to be converted into ngrams
    :param n: int    : number of ngrams for which representation is calculated
    :return: term frequency vector for occuring ngrams
    """
    words = s.split()
    counts = defaultdict(int)
    for k in range(1,n+1):
        for i in range(len(words)-k+1):
            ngram = tuple(words[i:i+k])
            counts[ngram] += 1
    return counts

def cook_refs(refs, n=4): ## lhuang: oracle will call with "average"
    '''Takes a list of reference sentences for a single segment
    and returns an object that encapsulates everything that BLEU
    needs to know about them.
    :param refs: list of string : reference sentences for some image
    :param n: int : number of ngrams for which (ngram) representation is calculated
    :return: result (list of dict)
    '''
    return [precook(ref, n) for ref in refs]

def cook_test(test, n=4):
    '''Takes a test sentence and returns an object that
    encapsulates everything that BLEU needs to know about it.
    :param test: list of string : hypothesis sentence for some image
    :param n: int : number of ngrams for which (ngram) representation is calculated
    :return: result (dict)
    '''
    return precook(test, n, True)

class CiderScorer(object):
    """CIDEr scorer.
    """

    def copy(self):
        ''' copy the refs.'''
        new = CiderScorer(n=self.n)
        new.ctest = copy.copy(self.ctest)
        new.crefs = copy.copy(self.crefs)
        return new

    def __init__(self, test=None, refs=None, n=4, sigma=6.0):
        ''' singular instance '''
        self.n = n
        self.sigma = sigma
        self.crefs = []
        self.ctest = []
        self.document_frequency = defaultdict(float)
        self.cook_append(test, refs)
        self.ref_len = None

    def cook_append(self, test, refs):
        '''called by constructor and __iadd__ to avoid creating new instances.'''

        if refs is not None:
            self.crefs.append(cook_refs(refs))
            if test is not None:
                self.ctest.append(cook_test(test)) ## N.B.: -1
            else:
                self.ctest.append(None) # lens of crefs and ctest have to match

    def size(self):
        assert len(self.crefs) == len(self.ctest), "refs/test mismatch! %d<>%d" % (len(self.crefs), len(self.ctest))
        return len(self.crefs)

    def __iadd__(self, other):
        '''add an instance (e.g., from another sentence).'''

        if type(other) is tuple:
            ## avoid creating new CiderScorer instances
            self.cook_append(other[0], other[1])
        else:
            self.ctest.extend(other.ctest)
            self.crefs.extend(other.crefs)

        return self
    def compute_doc_freq(self):
        '''
        Compute term frequency for reference data.
        This will be used to compute idf (inverse document frequency later)
        The term frequency is stored in the object
        :return: None
        '''
        for refs in self.crefs:
            # refs, k ref captions of one image
            for ngram in set([ngram for ref in refs for (ngram,count) in ref.items()]):
                self.document_frequency[ngram] += 1
            # maxcounts[ngram] = max(maxcounts.get(ngram,0), count)

    def compute_cider(self, df_mode):
        def counts2vec(cnts):
            """
            Function maps counts of ngram to vector of tfidf weights.
            The function returns vec, an array of dictionary that store mapping of n-gram and tf-idf weights.
            The n-th entry of array denotes length of n-grams.
            :param cnts:
            :return: vec (array of dict), norm (array of float), length (int)
            """
            vec = [defaultdict(float) for _ in range(self.n)]
            length = 0
            norm = [0.0 for _ in range(self.n)]
            for (ngram,term_freq) in cnts.items():
                # give word count 1 if it doesn't appear in reference corpus
                df = np.log(max(1.0, self.document_frequency[ngram]))
                # ngram index
                n = len(ngram)-1
                # tf (term_freq) * idf (precomputed idf) for n-grams
                vec[n][ngram] = float(term_freq)*(self.ref_len - df)
                # compute norm for the vector.  the norm will be used for computing similarity
                norm[n] += pow(vec[n][ngram], 2)

                if n == 1:
                    length += term_freq
            norm = [np.sqrt(n) for n in norm]
            return vec, norm, length

        def sim(vec_hyp, vec_ref, norm_hyp, norm_ref, length_hyp, length_ref):
            '''
            Compute the cosine similarity of two vectors.
            :param vec_hyp: array of dictionary for vector corresponding to hypothesis
            :param vec_ref: array of dictionary for vector corresponding to reference
            :param norm_hyp: array of float for vector corresponding to hypothesis
            :param norm_ref: array of float for vector corresponding to reference
            :param length_hyp: int containing length of hypothesis
            :param length_ref: int containing length of reference
            :return: array of score for each n-grams cosine similarity
            '''
            delta = float(length_hyp - length_ref)
            # measure consine similarity
            val = np.array([0.0 for _ in range(self.n)])
            for n in range(self.n):
                # ngram
                for (ngram,count) in vec_hyp[n].items():
                    # vrama91 : added clipping
                    val[n] += min(vec_hyp[n][ngram], vec_ref[n][ngram]) * vec_ref[n][ngram]

                if (norm_hyp[n] != 0) and (norm_ref[n] != 0):
                    val[n] /= (norm_hyp[n]*norm_ref[n])

                assert(not math.isnan(val[n]))
                # vrama91: added a length based gaussian penalty
                val[n] *= np.e**(-(delta**2)/(2*self.sigma**2))
            return val

        # compute log reference length
        if df_mode == "corpus":
            self.ref_len = np.log(float(len(self.crefs)))
        elif df_mode == "coco-val-df":
            # if coco option selected, use length of coco-val set
            self.ref_len = np.log(float(40504))

        scores = []
        for test, refs in zip(self.ctest, self.crefs):
            # compute vector for test captions
            vec, norm, length = counts2vec(test)
            # compute vector for ref captions
            score = np.array([0.0 for _ in range(self.n)])
            for ref in refs:
                vec_ref, norm_ref, length_ref = counts2vec(ref)
                score += sim(vec, vec_ref, norm, norm_ref, length, length_ref)
            # change by vrama91 - mean of ngram scores, instead of sum
            score_avg = np.mean(score)
            # divide by number of references
            score_avg /= len(refs)
            # multiply score by 10
            score_avg *= 10.0
            # append score of an image to the score list
            scores.append(score_avg)
        return scores

    def compute_score(self, df_mode, option=None, verbose=0):
        # compute idf
        if df_mode == "corpus":
            self.compute_doc_freq()
            # assert to check document frequency
            assert(len(self.ctest) >= max(self.document_frequency.values()))
            # import json for now and write the corresponding files
        else:
            self.document_frequency = pickle.load(open(os.path.join('data', df_mode + '.p'),'r'))
        # compute cider score
        score = self.compute_cider(df_mode)
        # debug
        # print score
        return np.mean(np.array(score)), np.array(score)
    
class CiderD:
    """
    Main Class to compute the CIDEr metric

    """
    def __init__(self, n=4, sigma=6.0, df="corpus"):
        # set cider to sum over 1 to 4-grams
        self._n = n
        # set the standard deviation parameter for gaussian penalty
        self._sigma = sigma
        # set which where to compute document frequencies from
        self._df = df

    def compute_score(self, gts, res):
        """
        Main function to compute CIDEr score
        :param  hypo_for_image (dict) : dictionary with key <image> and value <tokenized hypothesis / candidate sentence>
                ref_for_image (dict)  : dictionary with key <image> and value <tokenized reference sentence>
        :return: cider (float) : computed CIDEr score for the corpus
        """

        cider_scorer = CiderScorer(n=self._n)

        for res_id in res:

            hypo = res_id['caption']
            ref = gts[res_id['image_id']]

            # Sanity check.
            assert(type(hypo) is list)
            assert(len(hypo) == 1)
            assert(type(ref) is list)
            assert(len(ref) > 0)
            cider_scorer += (hypo[0], ref)

        (score, scores) = cider_scorer.compute_score(self._df)

        return score, scores

    def method(self):
        return "CIDEr-D"

### IV. Data Preparation and Preprocessing

In [15]:
if not RESUME:
    imgid2captions = parse_tokens(CAPTION_FILE_PATH)

    all_captions = []
    for captions in imgid2captions.values():
        all_captions.extend(captions)

    vocab = Vocabulary(freq_threshold = MIN_WORD_FREQ)
    vocab.build_vocabulary(all_captions)

    with open(VOCAB_PATH, "wb") as f:
        pickle.dump(vocab, f)
    
    print("Vocabulary saved to:", VOCAB_PATH)

    vocab_size = len(vocab)
    print("Vocabulary size:", vocab_size)

    img_ids = list(imgid2captions.keys())
    random.shuffle(img_ids)

    split_idx = int(0.8 * len(img_ids))
    train_ids = img_ids[:split_idx]
    valtest_ids = img_ids[split_idx:]

    mid_idx = len(valtest_ids) // 2
    val_ids = valtest_ids[:mid_idx]
    test_ids = valtest_ids[mid_idx:]

    train_dict = {iid: imgid2captions[iid] for iid in train_ids}
    val_dict = {iid: imgid2captions[iid] for iid in val_ids}
    test_dict = {iid: imgid2captions[iid] for iid in test_ids}
else:
    with open(VOCAB_PATH, "rb") as f:
        vocab = pickle.load(f)
    
    vocab_size = len(vocab)
    print("Resuming training. Vocabulary size:", vocab_size)

    imgid2captions = parse_tokens(CAPTION_FILE_PATH)

    img_ids = list(imgid2captions.keys())
    random.shuffle(img_ids)

    split_idx = int(0.8 * len(img_ids))
    train_ids = img_ids[:split_idx]
    valtest_ids = img_ids[split_idx:]

    mid_idx = len(valtest_ids) // 2
    val_ids = valtest_ids[:mid_idx]
    test_ids = valtest_ids[mid_idx:]

    train_dict = {iid: imgid2captions[iid] for iid in train_ids}
    val_dict = {iid: imgid2captions[iid] for iid in val_ids}
    test_dict = {iid: imgid2captions[iid] for iid in test_ids}

Resuming training. Vocabulary size: 2982


In [16]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p = 0.5),
    transforms.ColorJitter(brightness = 0.2, contrast = 0.2, saturation = 0.2, hue = 0.1),
    # transforms.RandomResizedCrop(224, scale = (0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]),
    # transforms.RandomErasing(p = 0.5, scale = (0.02,0.2), ratio = (0.3,3.3)),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

In [17]:
train_dataset = FlickrDataset(train_dict, vocab, transform = train_transform)
val_dataset = FlickrDataset(val_dict, vocab, transform = val_transform)
test_dataset = FlickrDataset(test_dict, vocab, transform = val_transform)

In [18]:
train_loader = DataLoader(
    train_dataset, 
    batch_size = BATCH_SIZE, 
    shuffle = True, 
    collate_fn = collate_fn,
    drop_last = False,
    num_workers = NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset, 
    batch_size = BATCH_SIZE, 
    shuffle = False, 
    collate_fn = collate_fn,
    drop_last = False,
    num_workers = NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset, 
    batch_size = BATCH_SIZE, 
    shuffle = False, 
    collate_fn = collate_fn,
    drop_last = False,
    num_workers = NUM_WORKERS
)

### V. Training Process

In [19]:
encoder = Encoder(EMBED_DIM, freeze = False)
decoder = Decoder(EMBED_DIM, HIDDEN_DIM, vocab_size)
model = Model(encoder, decoder).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {total_params}")

total_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_trainable_params}")

You are using a model of type vit_mae to instantiate a model of type vit. This is not supported for all configurations of models and can yield errors.
Some weights of ViTModel were not initialized from the model checkpoint at facebook/vit-mae-base and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total model parameters: 93805222
Total trainable parameters: 37102246


In [20]:
criterion = nn.CrossEntropyLoss(ignore_index = vocab.stoi["<PAD>"])
parameters = list(model.decoder.parameters()) + list(model.encoder.linear.parameters())
optimizer = optim.AdamW([
    {"params": model.encoder.parameters(), "lr": LEARNING_RATE_ENCODER, "weight_decay": WEIGHT_DECAY_ENCODER},
    {"params": model.decoder.parameters(), "lr": LEARNING_RATE_DECODER, "weight_decay": WEIGHT_DECAY_DECODER}
])

In [21]:
start_epoch = 0
best_val_loss = float("inf")

In [22]:
if RESUME and os.path.exists(BEST_CHECKPOINT_PATH):
    print("Resuming from checkpoint:", BEST_CHECKPOINT_PATH)
    checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location = DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]
    print(f"Resumed from epoch {start_epoch} with best validation loss {best_val_loss:.4f}")
elif RESUME:
    print(f"Warning: Checkpoint file {BEST_CHECKPOINT_PATH} not found. Starting training from scratch.")

Resuming from checkpoint: ../models/vit_lstm_best-v4.pth
Resumed from epoch 4 with best validation loss 3.0506


In [ ]:
# train_loss_history = []
# val_loss_history = []
# patience = 0

# try:
#     for epoch in range(start_epoch, EPOCHS):
#         train_loss = train_one_epoch(model, train_loader, criterion, optimizer, vocab_size, epoch)
#         val_loss = validate(model, val_loader, criterion, vocab_size)

#         print(f"Epoch [{epoch+1}/{EPOCHS}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

#         if val_loss < best_val_loss:
#             best_val_loss = val_loss

#             checkpoint_dict = {
#                 "epoch": epoch,
#                 "model_state_dict": model.state_dict(),
#                 "optimizer_state_dict": optimizer.state_dict(),
#                 "best_val_loss": best_val_loss
#             }
#             torch.save(checkpoint_dict, BEST_CHECKPOINT_PATH)
#             print(f"New best model saved at {BEST_CHECKPOINT_PATH} with validation loss: {best_val_loss:.4f}")
#             patience = 0
#         else: 
#             patience += 1
#             if patience >= EARLY_STOPPING_PATIENCE:
#                 print("Early stopping triggered.")
#                 break

#         final_checkpoint_dict = {
#             "model_state_dict": model.state_dict()
#         }
#         torch.save(final_checkpoint_dict, FINAL_MODEL_PATH)

#         train_loss_history.append(train_loss)
#         val_loss_history.append(val_loss)
        

# except KeyboardInterrupt:
#     print("Training interrupted. Best checkpoint is already saved.")

# print(f"\Final model weights saved to {FINAL_MODEL_PATH}")
# print(f"Best val_loss={best_val_loss:.4f} (checkpoint at {BEST_CHECKPOINT_PATH})")


Epoch 1: 100%|██████████| 506/506 [32:38<00:00,  3.87s/batch, loss=4.0409]


Epoch [1/10], Train Loss: 4.6102, Val Loss: 3.9868
New best model saved at ../models/vit_lstm_best-v4.pth with validation loss: 3.9868


Epoch 2: 100%|██████████| 506/506 [32:36<00:00,  3.87s/batch, loss=3.5223]


Epoch [2/10], Train Loss: 3.7023, Val Loss: 3.4802
New best model saved at ../models/vit_lstm_best-v4.pth with validation loss: 3.4802


Epoch 3: 100%|██████████| 506/506 [32:35<00:00,  3.86s/batch, loss=3.1335]


Epoch [3/10], Train Loss: 3.3162, Val Loss: 3.2113
New best model saved at ../models/vit_lstm_best-v4.pth with validation loss: 3.2113


Epoch 4: 100%|██████████| 506/506 [32:21<00:00,  3.84s/batch, loss=2.8098]


Epoch [4/10], Train Loss: 3.0823, Val Loss: 3.0506
New best model saved at ../models/vit_lstm_best-v4.pth with validation loss: 3.0506


Epoch 5:  63%|██████▎   | 317/506 [20:35<12:27,  3.96s/batch, loss=2.9948]

Kena interupsi :"


In [23]:
train_loss_history = []
val_loss_history = []
patience = 0

try:
    for epoch in range(start_epoch, EPOCHS):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, vocab_size, epoch)
        val_loss = validate(model, val_loader, criterion, vocab_size)

        print(f"Epoch [{epoch+1}/{EPOCHS}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            checkpoint_dict = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss
            }
            torch.save(checkpoint_dict, BEST_CHECKPOINT_PATH)
            print(f"New best model saved at {BEST_CHECKPOINT_PATH} with validation loss: {best_val_loss:.4f}")
            patience = 0
        else: 
            patience += 1
            if patience >= EARLY_STOPPING_PATIENCE:
                print("Early stopping triggered.")
                break

        final_checkpoint_dict = {
            "model_state_dict": model.state_dict()
        }
        torch.save(final_checkpoint_dict, FINAL_MODEL_PATH)

        train_loss_history.append(train_loss)
        val_loss_history.append(val_loss)
        

except KeyboardInterrupt:
    print("Training interrupted. Best checkpoint is already saved.")

print(f"\Final model weights saved to {FINAL_MODEL_PATH}")
print(f"Best val_loss={best_val_loss:.4f} (checkpoint at {BEST_CHECKPOINT_PATH})")


Epoch 5: 100%|██████████| 506/506 [29:02<00:00,  3.44s/batch, loss=2.8915]


Epoch [5/10], Train Loss: 2.9078, Val Loss: 2.9430
New best model saved at ../models/vit_lstm_best-v4.pth with validation loss: 2.9430


Epoch 6: 100%|██████████| 506/506 [31:24<00:00,  3.72s/batch, loss=2.7801]


Epoch [6/10], Train Loss: 2.7705, Val Loss: 2.8685
New best model saved at ../models/vit_lstm_best-v4.pth with validation loss: 2.8685


Epoch 7: 100%|██████████| 506/506 [31:48<00:00,  3.77s/batch, loss=2.5181]


...

Epoch [8/10], Train Loss: 2.5584, Val Loss: 2.7835
New best model saved at ../models/vit_lstm_best-v4.pth with validation loss: 2.7835


Epoch 9: 100%|██████████| 506/506 [32:02<00:00,  3.80s/batch, loss=2.1571]


Epoch [9/10], Train Loss: 2.5041, Val Loss: 2.7569
New best model saved at ../models/vit_lstm_best-v4.pth with validation loss: 2.7569


Epoch 10: 100%|██████████| 506/506 [32:39<00:00,  3.87s/batch, loss=2.3770]


Epoch [10/10], Train Loss: 2.4307, Val Loss: 2.7327
New best model saved at ../models/vit_lstm_best-v4.pth with validation loss: 2.7327
\Final model weights saved to ../models/vit_lstm_final-v4.pth
Best val_loss=2.7327 (checkpoint at ../models/vit_lstm_best-v4.pth)


In [ ]:
# plt.plot(train_loss_history, label = "Train Loss")
# plt.plot(val_loss_history, label = "Validation Loss")
# plt.xlabel("Epoch")
# plt.ylabel("Loss")
# plt.legend()
# plt.title("Training and Validation Loss Over Epochs")
# plt.show()

### VI. Evaluation and Inference

In [19]:
class Vocabulary:
    def __init__(self, freq_threshold = 5):
        self.freq_threshold = freq_threshold
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {v: k for k, v in self.itos.items()}
        self.index = 4

    def __len__(self):
        return len(self.itos)
    
    def tokenizer(self, text):
        text = text.lower()
        tokens = re.findall(r"\w+", text)
        return tokens
    
    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        for sentence in sentence_list:
            tokens = self.tokenizer(sentence)
            frequencies.update(tokens)
        
        for word, freq in frequencies.items():
            if freq >= self.freq_threshold:
                self.stoi[word] = self.index
                self.itos[self.index] = word
                self.index += 1

    def numericalize(self, text):
        tokens = self.tokenizer(text)
        numericalized = []
        for token in tokens:
            if token in self.stoi:
                numericalized.append(self.stoi[token])
            else:
                numericalized.append(self.stoi["<UNK>"])

        return numericalized

In [20]:
EMBED_DIM = 256
HIDDEN_DIM = 512
MAX_SEQ_LEN = 50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCABULARY_SAVE_PATH = "../models/vocab-v4.pkl"
MODEL_SAVE_PATH = "../models/vit_lstm_best-v4.pth"

In [21]:
with open(VOCABULARY_SAVE_PATH, "rb") as f:
    vocab = pickle.load(f)

vocab_size = len(vocab)
print("Vocabulary size:", vocab_size)

Vocabulary size: 2982


In [22]:
class Encoder(nn.Module):
    def __init__(self, embed_dim, freeze = False):
        super().__init__()
        self.vit = ViTModel.from_pretrained(ENCODER)

        for param in self.vit.parameters():
            param.requires_grad = False

        self.linear = nn.Sequential(
            nn.Linear(self.vit.config.hidden_size, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim, embed_dim),
            nn.LayerNorm(embed_dim)
        )

    def forward(self, images):
        with torch.no_grad():
            vit_outputs = self.vit(pixel_values = images)
            patch_embeddings = vit_outputs.last_hidden_state[:, 1:, :]

        features = self.linear(patch_embeddings)
        return features

In [23]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, encoder_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim
        self.head_dim = hidden_dim // num_heads

        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"

        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key = nn.Linear(encoder_dim, hidden_dim)
        self.value = nn.Linear(encoder_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, encoder_dim)

    def forward(self, hidden, encoder_outputs):
        B, N, _ = encoder_outputs.shape

        Q = self.query(hidden).view(B, self.num_heads, self.head_dim)    # [B, heads, head_dim]
        K = self.key(encoder_outputs).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # [B, heads, N, head_dim]
        V = self.value(encoder_outputs).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q.unsqueeze(2), K.transpose(-2, -1)) / (self.head_dim ** 0.5)  # [B, heads, 1, N]
        attn = torch.softmax(scores, dim=-1)  

        context = torch.matmul(attn, V)  
        context = context.transpose(1, 2).contiguous().view(B, self.hidden_dim)  # [B, H]

        return self.fc_out(context)  

class Decoder(nn.Module):
    def __init__(self, embed_dim, hidden_dim, vocab_size, encoder_dim = 256, num_layers = 2, dropout = 0.3, num_heads = 4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(embed_dim + encoder_dim, hidden_dim, num_layers, batch_first = True, dropout = dropout if num_layers > 1 else 0)
        self.attention = MultiHeadAttention(hidden_dim, encoder_dim, num_heads = num_heads)

        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, encoder_outputs, captions):
        captions_in = captions[:, :-1]  
        emb = self.dropout(self.embedding(captions_in))

        batch_size, seq_len, _ = emb.size()
        h, c = (torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size, device=emb.device),
                torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size, device=emb.device))

        outputs = []
        for t in range(seq_len):
            context = self.attention(h[-1], encoder_outputs)  
            lstm_input = torch.cat((emb[:, t], context), dim=1).unsqueeze(1)  
            out, (h, c) = self.lstm(lstm_input, (h, c))
            outputs.append(self.fc(out.squeeze(1)))

        outputs = torch.stack(outputs, dim=1)  
        return outputs
    
    def generate(self, features, max_len = 50, start_index = 1, end_index = 2, beam_size = 3, beam_search = True):
        if not beam_search:
            states = (torch.zeros(self.lstm.num_layers, features.size(0), self.lstm.hidden_size, device=features.device),
                    torch.zeros(self.lstm.num_layers, features.size(0), self.lstm.hidden_size, device=features.device))
            generated = []

            current_token = torch.LongTensor([start_index]).to(features.device).unsqueeze(0)

            for _ in range(max_len):
                emb = self.embedding(current_token).squeeze(1)  # [B, E]
                context = self.attention(states[0][-1], features)  # [B, D]
                lstm_input = torch.cat((emb, context), dim=1).unsqueeze(1)  # [B, 1, E+D]

                out, states = self.lstm(lstm_input, states)
                logits = self.fc(out.squeeze(1))

                predicted = logits.argmax(dim=1).item()
                generated.append(predicted)

                if predicted == end_index:
                    break

                current_token = torch.LongTensor([predicted]).to(features.device).unsqueeze(0)

            return generated    
        else:
            B = features.size(0)
            device = features.device

            # Initialize LSTM states
            states = (torch.zeros(self.lstm.num_layers, B, self.lstm.hidden_size, device=device),
                    torch.zeros(self.lstm.num_layers, B, self.lstm.hidden_size, device=device))

            # Each beam: (sequence, log_prob, states)
            beams = [([start_index], 0.0, states) for _ in range(beam_size)]

            for _ in range(max_len):
                new_beams = []

                for seq, log_prob, (h, c) in beams:
                    current_token = torch.LongTensor([seq[-1]]).to(device).unsqueeze(0)  # [1,1]
                    emb = self.embedding(current_token).squeeze(1)  # [1, E]
                    context = self.attention(h[-1], features)      # [1, D]
                    lstm_input = torch.cat((emb, context), dim=1).unsqueeze(1)  # [1,1,E+D]

                    out, (h_new, c_new) = self.lstm(lstm_input, (h, c))
                    logits = self.fc(out.squeeze(1))  # [1, vocab_size]
                    log_probs = torch.log_softmax(logits, dim=1)  # log probabilities

                    # Take top beam_size candidates for this token
                    top_log_probs, top_indices = log_probs.topk(beam_size, dim=1)

                    for k in range(beam_size):
                        next_seq = seq + [top_indices[0, k].item()]
                        next_log_prob = log_prob + top_log_probs[0, k].item()
                        new_beams.append((next_seq, next_log_prob, (h_new, c_new)))

                # Keep top beam_size sequences across all candidates
                new_beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]
                beams = new_beams

                # Stop early if all beams end with <EOS>
                if all(seq[-1] == end_index for seq, _, _ in beams):
                    break

            # Return the best sequence (highest log probability)
            best_seq = beams[0][0]

            # Remove start token if present
            if best_seq[0] == start_index:
                best_seq = best_seq[1:]

            return best_seq                

In [24]:
class Model(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def generate(self, images, max_len = MAX_SEQ_LEN):
        features = self.encoder(images)
        captions = self.decoder.generate(features, max_len = max_len, beam_search = True)
        return captions

In [25]:
best_model = Model(Encoder(EMBED_DIM, freeze = True), Decoder(EMBED_DIM, HIDDEN_DIM, vocab_size)).to(DEVICE)
best_model.load_state_dict(torch.load(BEST_CHECKPOINT_PATH, map_location = DEVICE)["model_state_dict"])

You are using a model of type vit_mae to instantiate a model of type vit. This is not supported for all configurations of models and can yield errors.
Some weights of ViTModel were not initialized from the model checkpoint at facebook/vit-mae-base and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<All keys matched successfully>

In [26]:
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

In [27]:
def generate_caption(image):
    pil_image = image.convert("RGB")
    image_tensor = inference_transform(pil_image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output_indices = best_model.generate(image_tensor, max_len = MAX_SEQ_LEN)

    result_words = []
    end_token_index = vocab.stoi["<EOS>"]
    for idx in output_indices:
        if idx == end_token_index:
            break
        word = vocab.itos.get(idx, "<UNK>")
        if word not in ("<SOS>", "<EOS>", "<PAD>"):
            result_words.append(word)

    caption = " ".join(result_words)
    return caption

In [28]:
generate_caption(Image.open(os.path.join(IMAGE_FOLDER_PATH, "12830823_87d2654e31.jpg")))

'a group of people are playing in a pool'

In [ ]:
candidate_list = []
reference_list = {}   # must be a dict

i = 0
for iid, references in test_dict.items():
    image_path = os.path.join(IMAGE_FOLDER_PATH, iid)
    image = Image.open(image_path)

    # Generate candidate caption (string)
    candidate = generate_caption(image)

    # Add to candidates (list of dicts)
    candidate_list.append({
        "image_id": i,
        "caption": [candidate]   # must be list of length 1
    })

    # Add to references (dict mapping int → list of strings)
    reference_list[i] = references   # references should already be a list of 5 captions

    i += 1

In [49]:
ciderD = CiderD()
final_ciderD = ciderD.compute_score(reference_list, candidate_list)[0]
final_ciderD = final_ciderD * 100 
print(f"Final CIDEr-D on test set: {final_ciderD:.4f}")

Final CIDEr-D on test set: 54.6271
